[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_02_exercise.ipynb)

> **Course repository:** Data and notebook files are loaded directly from the course GitHub repository.

# Module 5, NLP Exercise: Who Wrote It?

**Notebook:** `05_02_exercise`

## What this notebook is

In `05_01_main_classical`, we turned text into interpretable numeric features: lexical diversity, readability, sentence structure, grammatical features, and sentiment. The loan example had an important advantage: because the data were synthetic, we already knew where the signal lived.

This exercise reverses the problem.

You will work with passages from **The Federalist Papers**, written under the shared pseudonym *Publius*. Your task is to determine whether measurable characteristics of writing style can help distinguish passages written by **Alexander Hamilton** from passages written by **James Madison**.

The exercise is a form of **authorship attribution**, often called **stylometry** when it relies on measurable patterns in writing style.

## What we'll do

1. Load a labeled set of Federalist passages.
2. Build familiar classical NLP features such as TTR, readability, sentence length, and punctuation rates.
3. Test whether those obvious stylistic features identify the author.
4. Add **function-word frequencies**: small grammatical words writers often use without thinking about them.
5. Train an interpretable logistic-regression classifier.
6. Apply the final model to a set of previously unseen essays.

## A heads-up about the data

Unlike the loan example, the signal here was **not planted by us**. These are real historical texts. That makes model validation more important.

Each source essay has been divided into several passages. Passages from the same essay are related, so we will **never put passages from the same essay in both training and validation folds**. The `source_group` column exists only to enforce that rule. It is **not a predictor**.

## 0) Setup

We only need a lightweight classical-NLP stack:

- **NLTK** for word and sentence tokenization.
- **textstat** for readability scores.
- **scikit-learn** for logistic regression and evaluation.
- **pandas / seaborn / matplotlib** for analysis and visualization.

The first Colab run installs `textstat` and downloads the NLTK tokenizer resources.

In [ ]:
%pip install -q textstat nltk seaborn

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import textstat
from nltk.tokenize import word_tokenize, sent_tokenize

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_STATE = 1955

## 1) Load the Federalist training data

The training file contains **80 passages**:

- 40 Hamilton passages
- 40 Madison passages
- 16 source essays total
- 5 passages from each source essay

The important columns are:

- `passage_id`: a neutral passage identifier.
- `author`: the label we want to predict.
- `source_group`: identifies passages that came from the same original essay.
- `text`: the passage itself.

The data live in the course GitHub repository, so Colab can load them directly with `pandas`.

In [ ]:
# Course data location on GitHub.
DATA_BASE_URL = "https://raw.githubusercontent.com/tunnel-ai/way/main/nlp_data"

TRAINING_URL = f"{DATA_BASE_URL}/federalist_authorship_training_80.csv"
CHALLENGE_URL = f"{DATA_BASE_URL}/federalist_blind_challenge_student.csv"


# Load the labeled training corpus.
df = pd.read_csv(TRAINING_URL)

print(f"Rows: {len(df)}")
print(f"Source essays: {df['source_group'].nunique()}")
print("\nAuthor balance:")
print(df["author"].value_counts())

df.head()

### 1.1 Look at the raw text before modeling it

Before converting language into numbers, inspect a few passages.

**Question:** If you did not know the labels, what writing characteristics might you look for?

Possible hypotheses might include:

- sentence length,
- vocabulary complexity,
- readability,
- punctuation habits,
- repeated grammatical patterns.

Do **not** worry yet about whether those hypotheses are correct. The point is to turn them into measurable features and test them.

In [ ]:
pd.set_option("display.max_colwidth", 220)

df[["passage_id", "author", "text"]].sample(4, random_state=RANDOM_STATE)

### Why `source_group` matters

Suppose five passages all came from the same Federalist essay. If four appeared in the training set and the fifth appeared in the test set, the model would be seeing two very similar pieces of the same document.

That could make performance look better than it really is.

We want a harder question:

> **Can the model identify the author of an essay it has never seen before?**

So our validation folds will keep each `source_group` together. This is an example of preventing **data leakage**.

In [ ]:
pd.crosstab(df["source_group"], df["author"])

## 2) Build classical linguistic features

We will start with the kinds of features used in the previous notebook. These capture broad characteristics of how a passage is written.

We will build three families:

1. **Lexical diversity**: Type-Token Ratio (TTR)
2. **Readability and complexity**: Flesch-Kincaid, Gunning Fog, sentence length, word length
3. **Punctuation habits**: commas and semicolons per 100 words

The initial hypothesis is straightforward:

> Maybe Hamilton and Madison differ in how complex, varied, or syntactically dense their writing is.

### 2.1 Type-Token Ratio (TTR)

TTR measures lexical diversity:

$$\mathrm{TTR} = \frac{\text{unique words}}{\text{total words}}$$

Higher values indicate a more varied vocabulary.

**Caveat.** TTR changes with document length. Our passages are deliberately similar in length, which makes it more useful here than it would be across documents of wildly different sizes.

In [ ]:
def clean_words(text):
    """Lowercase alphabetic word tokens only."""
    return [w.lower() for w in word_tokenize(str(text)) if w.isalpha()]


def compute_ttr(text):
    words = clean_words(text)
    return len(set(words)) / len(words) if words else 0


df["ttr"] = df["text"].apply(compute_ttr)

df[["passage_id", "author", "ttr"]].head()

### 2.2 Readability scores

We will use the same general idea as the earlier notebook:

- **Flesch-Kincaid Grade Level** estimates the US school grade associated with the passage.
- **Gunning Fog Index** also uses sentence length and word complexity.

If one author consistently writes longer sentences or more complex words, these measures might capture it.

In [ ]:
df["flesch_kincaid"] = df["text"].apply(textstat.flesch_kincaid_grade)
df["gunning_fog"] = df["text"].apply(textstat.gunning_fog)

df[["passage_id", "author", "flesch_kincaid", "gunning_fog"]].head()

### 2.3 Surface complexity and punctuation

Next we add a few deliberately simple features:

- **Average sentence length**: words per sentence.
- **Sentence-length variability**: standard deviation of sentence lengths.
- **Average word length**: characters per word.
- **Comma rate**: commas per 100 words.
- **Semicolon rate**: semicolons per 100 words.

These features are crude, but that is part of the point. Classical NLP often starts with transparent measurements before moving to more elaborate representations.

In [ ]:
def avg_sentence_length(text):
    sentences = sent_tokenize(str(text))
    lengths = [len(clean_words(sentence)) for sentence in sentences]
    return np.mean(lengths) if lengths else 0


def sd_sentence_length(text):
    sentences = sent_tokenize(str(text))
    lengths = [len(clean_words(sentence)) for sentence in sentences]
    return np.std(lengths) if lengths else 0


def avg_word_length(text):
    words = clean_words(text)
    return np.mean([len(word) for word in words]) if words else 0


def punctuation_rate(text, mark):
    words = clean_words(text)
    return 100 * str(text).count(mark) / len(words) if words else 0


df["avg_sentence_length"] = df["text"].apply(avg_sentence_length)
df["sd_sentence_length"] = df["text"].apply(sd_sentence_length)
df["avg_word_length"] = df["text"].apply(avg_word_length)
df["comma_rate"] = df["text"].apply(lambda x: punctuation_rate(x, ","))
df["semicolon_rate"] = df["text"].apply(lambda x: punctuation_rate(x, ";"))

STYLE_COLS = [
    "ttr",
    "flesch_kincaid",
    "gunning_fog",
    "avg_sentence_length",
    "sd_sentence_length",
    "avg_word_length",
    "comma_rate",
    "semicolon_rate",
]

df[["author"] + STYLE_COLS].head()

### What did we just build?

The prose has become a small tabular fingerprint. Each passage now has eight transparent numeric descriptors.

Before fitting a classifier, compare the average values for Hamilton and Madison.

**Question:** Which features look different? Which look nearly identical?

In [ ]:
style_means = df.groupby("author")[STYLE_COLS].mean().T
style_means["difference_M_minus_H"] = style_means["Madison"] - style_means["Hamilton"]
style_means.round(3)

### 2.4 A quick visual check

Readability was one of our initial hypotheses. If readability strongly identifies the author, the distributions should separate visually.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x="author", y="flesch_kincaid", ax=axs[0])
axs[0].set_title("Flesch-Kincaid grade by author")

sns.boxplot(data=df, x="author", y="avg_sentence_length", ax=axs[1])
axs[1].set_title("Average sentence length by author")

plt.tight_layout()
plt.show()

## 3) Closing the loop: style features to an author

We now have numeric features and a labeled outcome. That means this is an ordinary supervised-learning problem.

We will use **logistic regression** for the same reason as in the previous notebook:

- it is fast,
- it works well as a baseline,
- and its coefficients remain interpretable.

But our validation strategy changes.

Instead of a random row-level split, we use **Stratified Group K-Fold cross-validation**. Every passage from a source essay stays in the same fold.

We will compare three representations:

1. **Readability only**
2. **Broader style features**
3. Later, **function words + style**

Chance accuracy is about **50%** because the corpus is balanced.

In [ ]:
READABILITY_COLS = ["flesch_kincaid", "gunning_fog"]

y = (df["author"] == "Madison").astype(int).to_numpy()
groups = df["source_group"].to_numpy()

base_model = Pipeline([
    ("scale", StandardScaler()),
    ("logit", LogisticRegression(max_iter=2000)),
])


def grouped_cv_predictions(feature_cols, n_splits=4):
    """Out-of-fold probabilities while keeping source essays intact."""
    X = df[feature_cols].to_numpy()
    probs = np.zeros(len(df))

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for train_idx, test_idx in cv.split(X, y, groups):
        model = clone(base_model)
        model.fit(X[train_idx], y[train_idx])
        probs[test_idx] = model.predict_proba(X[test_idx])[:, 1]

    preds = (probs >= 0.5).astype(int)
    return probs, preds


def evaluate_feature_set(name, feature_cols):
    probs, preds = grouped_cv_predictions(feature_cols)
    return {
        "feature_set": name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y, preds),
        "roc_auc": roc_auc_score(y, probs),
    }


results = [
    evaluate_feature_set("Readability only", READABILITY_COLS),
    evaluate_feature_set("Broader style", STYLE_COLS),
]

pd.DataFrame(results).round(3)

### What happened?

Do not jump immediately to "the model failed."

Instead ask:

1. Did readability perform much better than chance?
2. Did the broader style representation improve the result?
3. What does this tell us about our original hypothesis?

A weak result can be useful evidence. It tells us that **the representation we chose may not contain the signal we need**.

That brings us to a less obvious idea.

## 4) Function words: the boring words may be the useful ones

Writers make conscious choices about topics and important vocabulary. They may make much less conscious choices about tiny grammatical words such as:

> *the, and, of, to, by, on, not, there, upon...*

These are often called **function words** or grammatical words. They carry less topic content than nouns such as *army*, *commerce*, or *constitution*, but their habitual use can become part of a writer's stylistic fingerprint.

That makes them attractive for authorship attribution: ideally, we want to identify **how** someone writes rather than simply **what** they are writing about.

We will measure each function word as occurrences per 100 words.

In [ ]:
FUNCTION_WORDS = [
    "a", "an", "the",
    "and", "but", "or", "if", "that", "which",
    "of", "to", "in", "for", "from", "with", "by", "on", "at", "upon",
    "as", "than", "not", "there", "this", "these", "those", "such",
    "it", "its", "they", "them", "their", "we", "our", "you",
    "is", "are", "was", "were", "be", "been", "has", "have", "had",
    "may", "will", "would", "can", "must", "more", "all", "any", "also",
]


def function_word_rates(text):
    words = clean_words(text)
    counts = Counter(words)
    n = len(words)

    return {
        f"fw_{word}": 100 * counts[word] / n if n else 0
        for word in FUNCTION_WORDS
    }


function_df = pd.DataFrame(df["text"].apply(function_word_rates).tolist())
df = pd.concat([df.reset_index(drop=True), function_df], axis=1)

FUNCTION_COLS = list(function_df.columns)
ALL_COLS = STYLE_COLS + FUNCTION_COLS

print(f"Added {len(FUNCTION_COLS)} function-word features.")
df[["author"] + FUNCTION_COLS[:8]].head()

### 4.1 Which function words differ most?

This is descriptive analysis, not yet a model.

We will calculate the mean rate for each author and rank words by the absolute difference between Hamilton and Madison.

**Important:** A difference does not automatically make a word a reliable predictor. We still need out-of-sample validation.

In [ ]:
fw_means = df.groupby("author")[FUNCTION_COLS].mean().T
fw_means["difference_M_minus_H"] = fw_means["Madison"] - fw_means["Hamilton"]
fw_means["abs_difference"] = fw_means["difference_M_minus_H"].abs()

(
    fw_means
    .sort_values("abs_difference", ascending=False)
    .head(12)
    .round(3)
)

### 4.2 Re-run the classifier

Now compare the earlier representations with:

- function words alone,
- and the combined style + function-word representation.

If function words contain an authorship signal, grouped validation should improve even though the model has never seen the held-out essays.

In [ ]:
results = [
    evaluate_feature_set("Readability only", READABILITY_COLS),
    evaluate_feature_set("Broader style", STYLE_COLS),
    evaluate_feature_set("Function words", FUNCTION_COLS),
    evaluate_feature_set("Style + function words", ALL_COLS),
]

results_df = pd.DataFrame(results)
results_df[["feature_set", "n_features", "accuracy", "roc_auc"]].round(3)

### 4.3 Inspect the errors

Accuracy gives us one number. The confusion matrix tells us **which author the model confuses with which**.

The predictions below are still fully out-of-sample at the essay level.

In [ ]:
final_oof_probs, final_oof_preds = grouped_cv_predictions(ALL_COLS)

print(classification_report(
    y,
    final_oof_preds,
    target_names=["Hamilton", "Madison"],
    digits=3,
))

cm = confusion_matrix(y, final_oof_preds)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=["Hamilton", "Madison"],
    yticklabels=["Hamilton", "Madison"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Grouped out-of-sample confusion matrix")
plt.show()

### 4.4 Which features did the final model lean on?

To interpret the final model, fit it once on the full labeled corpus and inspect the standardized logistic-regression coefficients.

Because the predictors are z-scored, coefficient magnitudes are roughly comparable:

- **Positive** coefficients push toward **Madison**.
- **Negative** coefficients push toward **Hamilton**.

Do not interpret a coefficient as causal. It tells us which features help this classifier separate these authors in this corpus.

In [ ]:
final_model = clone(base_model)
final_model.fit(df[ALL_COLS], y)

coefs = pd.Series(
    final_model.named_steps["logit"].coef_[0],
    index=ALL_COLS,
)

top_features = coefs.loc[coefs.abs().sort_values(ascending=False).head(15).index]
top_features = top_features.sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
top_features.plot(kind="barh", ax=ax)
ax.axvline(0, linewidth=1)
ax.set_xlabel("Standardized logistic coefficient")
ax.set_title("Strongest features in the final authorship model")
plt.tight_layout()
plt.show()

top_features.sort_values(key=np.abs, ascending=False).to_frame("coefficient")

## 5) Blind challenge: five unseen essays

Now we close the loop.

The challenge file contains **five groups of passages** labeled only `Q01` through `Q05`. None of these essays appears in the training corpus.

Your job is to use the model you already built. **Do not change the feature set or tune the model after looking at the challenge predictions.** That would turn the challenge set into another training set.

For each passage, the model will produce a probability-like score for Madison. We will then aggregate the five passages belonging to the same essay.

Why aggregate?

A single passage can be noisy. Several passages give us multiple pieces of evidence about the same underlying author.

In [ ]:
challenge = pd.read_csv(CHALLENGE_URL)

print(f"Challenge passages: {len(challenge)}")
print(f"Challenge essays: {challenge['challenge_group'].nunique()}")
print("\nPassages per essay:")
print(challenge["challenge_group"].value_counts().sort_index())

challenge.head()

### 5.1 Build the same features for the challenge set

This step is important: **training and challenge data must go through exactly the same feature-engineering pipeline**.

In [ ]:
challenge["ttr"] = challenge["text"].apply(compute_ttr)
challenge["flesch_kincaid"] = challenge["text"].apply(textstat.flesch_kincaid_grade)
challenge["gunning_fog"] = challenge["text"].apply(textstat.gunning_fog)
challenge["avg_sentence_length"] = challenge["text"].apply(avg_sentence_length)
challenge["sd_sentence_length"] = challenge["text"].apply(sd_sentence_length)
challenge["avg_word_length"] = challenge["text"].apply(avg_word_length)
challenge["comma_rate"] = challenge["text"].apply(lambda x: punctuation_rate(x, ","))
challenge["semicolon_rate"] = challenge["text"].apply(lambda x: punctuation_rate(x, ";"))

challenge_function_df = pd.DataFrame(
    challenge["text"].apply(function_word_rates).tolist()
)
challenge = pd.concat([challenge.reset_index(drop=True), challenge_function_df], axis=1)

challenge[["challenge_passage_id", "challenge_group"] + STYLE_COLS].head()

### 5.2 Predict each passage, then aggregate by essay

The model was already fit on the complete labeled training corpus in Section 4.4.

We will produce two pieces of evidence for each mystery essay:

1. **Madison votes**: how many of its five passages were classified as Madison?
2. **Mean Madison score**: the average model score across all five passages.

Treat these scores as model evidence, **not as perfectly calibrated historical probabilities**.

In [ ]:
challenge["prob_madison"] = final_model.predict_proba(
    challenge[ALL_COLS]
)[:, 1]

challenge["predicted_author"] = np.where(
    challenge["prob_madison"] >= 0.5,
    "Madison",
    "Hamilton",
)

challenge_summary = (
    challenge
    .groupby("challenge_group")
    .agg(
        passages=("challenge_passage_id", "count"),
        madison_votes=("predicted_author", lambda x: (x == "Madison").sum()),
        mean_madison_score=("prob_madison", "mean"),
        median_madison_score=("prob_madison", "median"),
    )
)

challenge_summary["predicted_author"] = np.where(
    challenge_summary["mean_madison_score"] >= 0.5,
    "Madison",
    "Hamilton",
)

challenge_summary.round(3)

## 6) Your interpretation

Before your instructor reveals anything about the five essays, answer these questions.

### Model-building questions

1. How well did **readability alone** distinguish Hamilton from Madison?
2. What happened when you added broader stylistic features?
3. What happened when you added **function words**?
4. Which individual features had the largest standardized coefficients?
5. Why might a word such as *upon*, *not*, or *there* be useful for authorship attribution even though it carries little topic meaning?

### Validation questions

6. Why did we group passages by source essay during cross-validation?
7. What could go wrong if we randomly split all 80 passages into train and test sets?
8. Is accuracy alone enough to establish that the model has learned "authorship"? Why or why not?

### Blind-challenge questions

9. Record your predicted author for Q01–Q05.
10. Which predictions are strongest? Which are most uncertain?
11. Did all five passages from every essay agree? If not, what does that tell you?
12. What evidence would make you more confident that the model is identifying **style** rather than simply identifying **topic**?

## What we built

Starting from historical prose, we ended up with:

- **Original text and labels**: Hamilton vs. Madison.
- **Classical linguistic features**: TTR, readability, sentence structure, word length, punctuation.
- **Stylometric features**: rates of common function words.
- **A grouped validation design** that prevents passages from the same essay from leaking across training and validation.
- **An interpretable logistic regression** that maps writing characteristics to authorship.
- **A blind application** to five completely unseen essays.

The broader lesson is the same one from the previous notebook:

> **Match the representation to where the signal lives.**

In the loan example, the strongest signal was largely lexical and sentiment-related. Here, the useful signal may be much more subtle: repeated stylistic habits that writers themselves may not consciously notice.

That is the basic logic of **stylometry**: turn patterns of language into measurable features, then test whether those patterns generalize to new text.